# 🔄 Polyglot Code Translator
**TypeScript → Python & Go with HF-Based Quality Metrics & Analysis**

_End-to-End Code Translation — LLM + Hugging Face for Translation Quality, Security, Complexity & Idiomaticity_

---

## What I Learned in Week 4

Week 4 covered the core building blocks of code generation using frontier LLMs. This notebook extends those ideas with:
- **Code generation** — Frontier LLM translates TS → Python and Go (course used Python → C++; here TS → Python/Go)
- **Prompt engineering** — System prompts for translation and for **Markdown-formatted analysis** so Gradio can render it directly
- **HF for language detection** — `CodeBERTa-language-id` with TypeScript vs JavaScript disambiguation and confidence scores
- **HF for evaluation** — Translation quality (CodeBERT similarity), code security, structural complexity, idiomaticity (stack-edu classifiers) instead of execution-time benchmarks
- **Multi-model support** — OpenAI-compatible client (GPT, Anthropic, Ollama)
- **Gradio UX** — Tabs for stats, progress bar for long runs, side-by-side code output

## Why This Project

As a TypeScript/Node.js developer learning Python, I wanted a tool that:
1. Translates my existing TS knowledge to Python/Go
2. Shows equivalent code for OOP concepts (classes, inheritance, design patterns)
3. **Evaluates** translations with HF models (similarity, security, complexity, idiomaticity) rather than raw execution timing
4. Provides context-aware translation and **structured analysis** (issues, performance tips, best practices) rendered as Markdown

This exercise combines Week 4 concepts with Hugging Face transformers for both detection and evaluation.

---

## Components Used

| Component | Purpose |
|---|---|
| OpenAI-compatible client | Connect to multiple LLM providers (GPT, Anthropic, Ollama) |
| `transformers` pipeline / models | Language detection (CodeBERTa-language-id), CodeBERT similarity, security classifier, stack-edu idiomaticity |
| System prompt engineering | Translation prompts + **Markdown-only** analysis prompt for clean rendering |
| `gr.Blocks`, `gr.Code`, `gr.Tabs`, `gr.Progress` | Interactive UI: code I/O, tabbed stats, progress feedback |
| Heuristic (keywords/lines) | Structural complexity score when no HF complexity model is used |

---

## How It Works

1. **Input** — User pastes TypeScript/JavaScript code
2. **Language detection** — HF CodeBERTa-language-id (TypeScript vs JavaScript + confidence)
3. **Analysis** — LLM returns Markdown (Issues, Performance tips, Best practices); Gradio renders it in the **Language & analysis** tab
4. **Translation** — LLM generates Python and Go; progress bar updates during this phase
5. **Metrics** — HF models run in sequence: translation similarity (CodeBERT), code quality/security, complexity, idiomaticity; results shown in **Stats** tabs
6. **Output** — Side-by-side Python/Go code + tabbed Stats (Language & analysis, Translation quality, Code quality, Complexity, Idiomaticity)

---

## Simple vs Advanced Examples

The tool handles both:

### Simple Examples
- Arrow functions → Python lambdas / Go funcs
- `filter/map/reduce` → List comprehensions / Go loops
- `const/let` → Python variables / Go var

### Advanced Examples
- Classes with inheritance
- Decorators/Annotations
- Interfaces → Python Protocols / Go interfaces
- Generics
- Design patterns (Factory, Singleton, Observer)

---

**Hardware:** Runs on CPU with optional GPU acceleration for embeddings and classification models.

## Cell 1 — Install Dependencies

In [ ]:
# Run once — restart kernel after installation
!uv pip install -q transformers torch gradio openai python-dotenv

print("✅ Dependencies installed. Restart kernel if needed.")

## Cell 2 — Imports & Configuration

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from transformers import pipeline

# Load environment
load_dotenv(override=True)

# API Keys
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key: {openai_api_key[:8]}...")
else:
    print("⚠️ No OpenAI key - using Ollama fallback")

if anthropic_api_key:
    print(f"Anthropic API Key: {anthropic_api_key[:7]}...")

# Model configuration
MODEL_CHOICE = "gpt"  # "gpt", "anthropic", or "ollama"
GPT_MODEL = "gpt-4.1-mini"
ANTHROPIC_MODEL = "claude-sonnet-4-5-20250529"

# Initialize clients
openai_client = OpenAI()
anthropic_client = OpenAI(
    base_url="https://api.anthropic.com/v1",
    api_key=anthropic_api_key
)
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

print("✅ Clients configured.")

## Cell 3 — Language Detection (HuggingFace)

In [ ]:
# Language detection for code via HuggingFace CodeBERTa-language-id

print("Loading language detection model...")

language_detector = pipeline(
    "text-classification",
    model="huggingface/CodeBERTa-language-id",
    top_k=1,
)

def _is_likely_typescript(code: str) -> bool:
    ts_markers = ("interface ", " type ", ": string", ": number", ": boolean", ": void", " as ", "enum ")
    return any(m in code for m in ts_markers)

def detect_language(code: str) -> dict:
    """Detect programming language and confidence. Returns {\"language\": str, \"confidence\": float}."""
    if not code.strip():
        return {"language": "unknown", "confidence": 0.0}
    snippet = code.strip()[:4096]
    out = language_detector(snippet)

    # Handle the case where the output is a list of lists
    if out and isinstance(out[0], list):
        out = out[0]

    # Handle the case where the output is a list of dictionaries
    if not out or len(out) == 0:
        return {"language": "unknown", "confidence": 0.0}
    pred = out[0]
    label = pred["label"].lower()
    confidence = float(pred["score"])
    if label == "javascript":
        language = "typescript" if _is_likely_typescript(snippet) else "javascript"
    else:
        language = label
    return {"language": language, "confidence": confidence}

# Test detection
test_code = """
interface User {
  name: string;
  age: number;
}
const getActiveUsers = (users: User[]) => users.filter(u => u.age > 18);
"""

result = detect_language(test_code)
print(f"Detected: {result['language']} (confidence: {result['confidence']:.2%})")
print("✅ Language detection ready.")

## Cell 4 — LLM Translation Functions

In [ ]:
def get_client_and_model():
    """Return the appropriate client and model based on choice."""
    if MODEL_CHOICE == "gpt":
        return openai_client, GPT_MODEL
    elif MODEL_CHOICE == "anthropic":
        return anthropic_client, ANTHROPIC_MODEL
    else:
        return ollama_client, "llama3.2"


def translate_code(code: str, target_lang: str = "python") -> str:
    """
    Translate TypeScript code to target language.
    
    Args:
        code: TypeScript/JavaScript source code
        target_lang: "python" or "go"
    
    Returns:
        Translated code in target language
    """
    client, model = get_client_and_model()
    
    # Build the prompt based on target language
    if target_lang == "python":
        system_prompt = """You are an expert code translator. Translate TypeScript/JavaScript code to idiomatic Python.
        
        IMPORTANT RULES:
        1. Use Pythonic patterns (list comprehensions, dataclasses for classes)
        2. Convert TypeScript interfaces to Python @dataclass or Protocol
        3. Convert arrow functions to regular functions or lambdas
        4. Use type hints where appropriate
        5. Maintain the same logic and behavior
        6. Add comments explaining equivalent Python patterns
        7. For classes, use proper Python OOP (not TypeScript-style)
        
        Provide ONLY the code, no explanations outside comments in the code."""
    else:  # Go
        system_prompt = """You are an expert code translator. Translate TypeScript/JavaScript code to idiomatic Go.
        
        IMPORTANT RULES:
        1. Use proper Go idioms (no classes, use structs with methods)
        2. Convert TypeScript interfaces to Go interfaces
        3. Use Go's error handling (no exceptions)
        4. Convert arrow functions to regular functions
        5. Use proper Go types (string, int, bool, etc.)
        6. Maintain the same logic and behavior
        7. Add comments explaining equivalent Go patterns
        
        Provide ONLY the code, no explanations outside comments in the code."""
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Translate this TypeScript code to {target_lang.title()}:\n\n{code}"}
    ]
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.3,
            max_tokens=2000
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"


print("✅ Translation functions ready.")

## Cell 5 — Translation quality & code metrics (HF)

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification

_quality_model = None
_quality_tokenizer = None
_security_model = None
_security_tokenizer = None
_idiom_models = {}
_idiom_tokenizers = {}

def _get_quality_model():
    global _quality_model, _quality_tokenizer
    if _quality_model is None:
        _quality_tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
        _quality_model = AutoModel.from_pretrained("microsoft/codebert-base")
        _quality_model.eval()
    return _quality_model, _quality_tokenizer

def translation_similarity(code_a: str, code_b: str, max_length: int = 512) -> float:
    """CodeBERT cosine similarity between two code snippets (0–1). Higher = more semantically similar."""
    try:
        model, tokenizer = _get_quality_model()
        with torch.no_grad():
            inp_a = tokenizer(code_a.strip()[:8000], return_tensors="pt", truncation=True, max_length=max_length, padding=True)
            inp_b = tokenizer(code_b.strip()[:8000], return_tensors="pt", truncation=True, max_length=max_length, padding=True)
            out_a = model(**inp_a).last_hidden_state[:, 0, :].squeeze()
            out_b = model(**inp_b).last_hidden_state[:, 0, :].squeeze()
            sim = torch.nn.functional.cosine_similarity(out_a.unsqueeze(0), out_b.unsqueeze(0)).item()
        return round((sim + 1) / 2, 4)
    except Exception:
        return 0.0

def code_quality_security(code: str) -> tuple:
    """HF model: secure (0) vs insecure (1). Returns (label, confidence)."""
    global _security_model, _security_tokenizer
    try:
        if _security_model is None:
            _security_tokenizer = AutoTokenizer.from_pretrained("mrm8488/codebert-base-finetuned-detect-insecure-code")
            _security_model = AutoModelForSequenceClassification.from_pretrained("mrm8488/codebert-base-finetuned-detect-insecure-code")
            _security_model.eval()
        inp = _security_tokenizer(code.strip()[:4000], return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            logits = _security_model(**inp).logits.squeeze()
        probs = torch.softmax(logits, dim=0)
        idx = int(torch.argmax(probs).item())
        label = "insecure" if idx == 1 else "secure"
        return label, round(probs[idx].item(), 4)
    except Exception:
        return "unknown", 0.0

def complexity_heuristic(code: str) -> float:
    """Structural complexity 0-1 (branches, loops, length)."""
    import re
    code = code.strip()
    if not code:
        return 0.0
    keywords = len(re.findall(r"\b(if|else|elif|for|while|try|except|with|def|class|switch|case)\b", code, re.I))
    lines = max(1, len([l for l in code.splitlines() if l.strip()]))
    raw = min(1.0, (keywords * 0.05 + lines * 0.002))
    return round(raw, 4)

def idiomaticity_score(code: str, language: str) -> float:
    """HF Stack-Edu classifier: educational/idiomatic quality 0-5 -> 0-1."""
    global _idiom_models, _idiom_tokenizers
    repo = {"python": "HuggingFaceTB/stack-edu-classifier-python", "go": "HuggingFaceTB/stack-edu-classifier-go", "typescript": "HuggingFaceTB/stack-edu-classifier-typescript"}.get(language.lower())
    if not repo or language.lower() not in ("python", "go", "typescript"):
        return 0.0
    try:
        if language.lower() not in _idiom_models:
            _idiom_tokenizers[language.lower()] = AutoTokenizer.from_pretrained(repo)
            _idiom_models[language.lower()] = AutoModelForSequenceClassification.from_pretrained(repo)
            _idiom_models[language.lower()].eval()
        tok = _idiom_tokenizers[language.lower()]
        model = _idiom_models[language.lower()]
        inp = tok(code.strip()[:6000], return_tensors="pt", truncation=True, max_length=1024)
        with torch.no_grad():
            logits = model(**inp).logits.squeeze()
        score = logits.item() if logits.dim() == 0 else logits[0].item()
        normalized = max(0, min(1, score / 5.0))
        return round(normalized, 4)
    except Exception:
        return 0.0

print("✅ Translation quality & code metrics (CodeBERT, security, complexity, idiomaticity) ready.")

## Cell 7 — Gradio UI

In [ ]:
def translate_pipeline(ts_code: str, progress=gr.Progress()):
    if not ts_code.strip():
        return "⚠️ Please enter TypeScript code", "", "", "", "", "", "", ts_code
    
    def report(p: float, msg: str):
        try: progress(p, desc=msg)
        except Exception: pass
    
    report(0.05, "Detecting language…")
    detected = detect_language(ts_code)
    report(0.30, "Translating to Python…")
    python_code = translate_code(ts_code, "python")
    report(0.45, "Translating to Go…")
    go_code = translate_code(ts_code, "go")
    
    report(0.50, "Translation similarity (Python)…")
    sim_py = translation_similarity(ts_code, python_code)
    report(0.55, "Translation similarity (Go)…")
    sim_go = translation_similarity(ts_code, go_code)
    report(0.60, "Code quality / security…")
    q_ts = code_quality_security(ts_code)
    q_py = code_quality_security(python_code)
    q_go = code_quality_security(go_code)
    report(0.72, "Complexity (structural)…")
    c_ts = complexity_heuristic(ts_code)
    c_py = complexity_heuristic(python_code)
    c_go = complexity_heuristic(go_code)
    report(0.85, "Idiomaticity (TS / Python / Go)…")
    i_ts = idiomaticity_score(ts_code, "typescript")
    i_py = idiomaticity_score(python_code, "python")
    i_go = idiomaticity_score(go_code, "go")
    report(1.0, "Done")
    
    tab_lang = f"**Language**: {detected['language']} (confidence: {detected['confidence']:.2%})\n\n---\n**Analysis**\n\n"
    tab_translation = f"**Translation quality (CodeBERT)**\n\nSemantic similarity to source (0–1):\n- source ↔ Python: **{sim_py}**\n- source ↔ Go: **{sim_go}**"
    tab_quality = f"**Code quality (security)**\n\n- Source: **{q_ts[0]}** ({q_ts[1]:.2f})\n- Python: **{q_py[0]}** ({q_py[1]:.2f})\n- Go: **{q_go[0]}** ({q_go[1]:.2f})"
    tab_complexity = f"**Complexity (structural, 0–1)**\n\n- Source: **{c_ts}**\n- Python: **{c_py}**\n- Go: **{c_go}**"
    tab_idiomaticity = f"**Idiomaticity (0–1, higher = more idiomatic)**\n\n- Source (TS): **{i_ts}**\n- Python: **{i_py}**\n- Go: **{i_go}**"
    return (tab_lang, tab_translation, tab_quality, tab_complexity, tab_idiomaticity, python_code, go_code, ts_code)


# Build Gradio Interface
with gr.Blocks(title="Polyglot Code Translator") as demo:
    gr.Markdown("# 🔄 Polyglot Code Translator\n\n**TypeScript → Python & Go with HF-Based Quality Metrics & Analysis**\n\n*End-to-End Code Translation — LLM + Hugging Face for Translation Quality, Security, Complexity & Idiomaticity*")

    with gr.Row():
        with gr.Column(scale=1):
            ts_input = gr.Code(
                label="Source code (TypeScript / JavaScript)",
                language="typescript",
                lines=18
            )
            translate_btn = gr.Button("Translate", variant="primary", size="lg")
            gr.Examples(
                examples=[
                    ["""interface User {
  name: string;
  age: number;
  status: 'active' | 'inactive';
}

const getActiveUsers = (users: User[]): User[] =>
  users.filter(user => user.status === 'active');
"""],
                    ["""class Calculator {
  private result: number = 0;

  add(value: number): this {
    this.result += value;
    return this;
  }

  getResult(): number {
    return this.result;
  }
}
"""],
                ],
                inputs=ts_input,
                label="Try an example",
            )

    gr.Markdown("---\n### Output")
    with gr.Tabs():
        with gr.Tab("Language & analysis"):
            tab_lang = gr.Markdown()
        with gr.Tab("Translation quality"):
            tab_translation = gr.Markdown()
        with gr.Tab("Code quality"):
            tab_quality = gr.Markdown()
        with gr.Tab("Complexity"):
            tab_complexity = gr.Markdown()
        with gr.Tab("Idiomaticity"):
            tab_idiomaticity = gr.Markdown()
    with gr.Row():
        python_output = gr.Code(
            label="Python",
            language="python",
            lines=18
        )
        go_output = gr.Code(
            label="Go",
            language="c",
            lines=18
        )

    translate_btn.click(
        translate_pipeline,
        inputs=[ts_input],
        outputs=[tab_lang, tab_translation, tab_quality, tab_complexity, tab_idiomaticity, python_output, go_output, ts_input],
    )

demo.launch(inbrowser=True)

---

## Concepts Demonstrated

| Cell | Pipeline / API | What it shows |
|------|---------------|---------------|
| 3 | Language detection | HF CodeBERTa-language-id; TypeScript vs JavaScript + confidence |
| 4 | Translation | System prompt engineering for TS → Python/Go |
| 4 | Analysis | LLM returns Markdown (Issues, Performance tips, Best practices); Gradio renders it |
| 5 | HF metrics | CodeBERT similarity, security classifier, complexity heuristic, stack-edu idiomaticity |
| 7 | Gradio UI | Tabs for stats, progress bar, side-by-side code, examples |
| 7 | Pipeline order | Translate first, then run HF metrics so user sees code sooner |

---

## Extensions Possible

1. **More languages**: Add Rust, Java, C# translations
2. **Go execution**: Optional execution-time benchmark for Go (in addition to HF metrics)
3. **Design patterns**: Special prompts for Factory, Singleton, etc.
4. **Diff view**: Show what changed between TS and Python/Go
5. **Save translations**: Store in Chroma or a DB for future reference
